# 1st LAB: Working with Documents

You can mount a google drive to connect you drive storage or start this notebook and temporarily add the file

> **BEWARE: if you use the temporary storage the file will be removed after the notebook is closed!**




In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

import pandas as pd

Mounted at /content/gdrive


In [ ]:
!ls gdrive/MyDrive/Labs/LAB1_TMS

data  Lab1_1_TMS_text_preprocessing.ipynb


# Importing the data

We use the **SMS Spam Collection** (`spam.csv`, from your Drive): 5,572 SMS in English.

| column | meaning | example |
|---|---|---|
| `v1` | label: `ham` = normal message, `spam` = advertising or scam | `spam` |
| `v2` | text of the SMS | `Free entry in 2 a wkly comp to win FA Cup final...` |
| `Unnamed: 2/3/4` | pieces of text split by mistake (see below) | `NaN` |

After cleaning we rename `v1` → `label` and `v2` → `text`.

We also keep all the SMS joined in a single string, `raw`, **before** any cleaning (used in the stop words section).

In [ ]:
## Reading the SMS Spam Collection from a csv file

# base_path = ''
base_path = 'gdrive/MyDrive/Labs/LAB1_TMS/data/' # use the path to your gdrive folder

data_spam = pd.read_csv(f"{base_path}spam.csv", encoding="ISO-8859-1")
display(data_spam.head())
print(data_spam.shape)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


(5572, 5)


In [ ]:
display(data_spam[data_spam['Unnamed: 2'].notna()].head())
print(data_spam[data_spam['Unnamed: 2'].notna()].shape)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
95,spam,Your free ringtone is waiting to be collected....,PO Box 5249,"MK17 92H. 450Ppw 16""",NaN
281,ham,\Wen u miss someone,the person is definitely special for u..... B...,why to miss them,"just Keep-in-touch\"" gdeve.."""
444,ham,\HEY HEY WERETHE MONKEESPEOPLE SAY WE MONKEYAR...,HOWU DOIN? FOUNDURSELF A JOBYET SAUSAGE?LOVE ...,NaN,NaN
671,spam,SMS. ac sun0819 posts HELLO:\You seem cool,"wanted to say hi. HI!!!\"" Stop? Send STOP to ...",NaN,NaN
710,ham,Height of Confidence: All the Aeronautics prof...,"this wont even start........ Datz confidence..""",NaN,NaN


(50, 5)


In [ ]:
# Filter rows where at least one of the Unnamed columns is not NaN
filtro_unnamed = data_spam['Unnamed: 2'].notna() | data_spam['Unnamed: 3'].notna() | data_spam['Unnamed: 4'].notna()

# Display the first few rows matching the filter and their total count
display(data_spam[filtro_unnamed].head())
print("Total number of rows with valued unnamed columns:", data_spam[filtro_unnamed].shape[0])

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
95,spam,Your free ringtone is waiting to be collected....,PO Box 5249,"MK17 92H. 450Ppw 16""",NaN
281,ham,\Wen u miss someone,the person is definitely special for u..... B...,why to miss them,"just Keep-in-touch\"" gdeve.."""
444,ham,\HEY HEY WERETHE MONKEESPEOPLE SAY WE MONKEYAR...,HOWU DOIN? FOUNDURSELF A JOBYET SAUSAGE?LOVE ...,NaN,NaN
671,spam,SMS. ac sun0819 posts HELLO:\You seem cool,"wanted to say hi. HI!!!\"" Stop? Send STOP to ...",NaN,NaN
710,ham,Height of Confidence: All the Aeronautics prof...,"this wont even start........ Datz confidence..""",NaN,NaN


Total number of rows with valued unnamed columns: 50


The columns `Unnamed: 2`, `Unnamed: 3` and `Unnamed: 4` are almost always empty.
In 50 rows, however, the text of the SMS contains commas and was split across them by the csv reader.

For example, row 95 is: `Your free ringtone is waiting to be collected....` + `PO Box 5249` + `MK17 92H. 450Ppw 16"`

If we kept only `v2` we would lose the end of these messages, so we join the pieces back together.

In [ ]:
# Join each message back into a single text: the csv was split on the commas
text_cols = ['v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']
data_spam['v2'] = data_spam[text_cols].apply(lambda row: ','.join(row.dropna()), axis=1)

# Keep only the label and the text, with readable names
data_spam = data_spam[['v1', 'v2']].rename(columns={'v1': 'label', 'v2': 'text'})
# Verify that messages have been correctly rebuilt
display(data_spam.loc[[95, 281]])
print(data_spam.shape)

,label,text
95,spam,Your free ringtone is waiting to be collected....
281,ham,"\Wen u miss someone, the person is definitely ..."


(5572, 2)


In [ ]:
import nltk
nltk.download('punkt') ###< run once
nltk.download('punkt_tab') ###< run once, needed by recent NLTK versions

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
# A copy of the SMS before any cleaning (used in the exercises at the end of the notebook)
sms_original = data_spam['text'].copy()

# All the SMS in a single string, one per line, before any cleaning.
# We join them into a single giant string ('raw') to count the tokens of the whole corpus
# (see the stop words section).
raw = "\n".join(data_spam['text'])
print('Number of characters:', len(raw))
print(raw[:300])

Number of characters: 455188
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...
Ok lar... Joking wif u oni...
Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's
U 


# Remove whitespaces

In [ ]:
print('Sample Text:\n')
text = "DEAR STUDENTS,\n\nTHE TEXT MINING LAB STARTS ON THE 10TH OF NOVEMBER 2026 IN U9.\n\n\nPLEASE BRING YOUR LAPTOP, AND INSTALL PYTHON \n\nBEST REGARDS,\n\n\t\t\t SEE YOU SOON"
print(text)

Sample Text:

DEAR STUDENTS,

THE TEXT MINING LAB STARTS ON THE 10TH OF NOVEMBER 2026 IN U9.


PLEASE BRING YOUR LAPTOP, AND INSTALL PYTHON 

BEST REGARDS,

			 SEE YOU SOON


In [ ]:
print('Initial text lenght:', len(text))
print("Number of 2 newlines", text.count('\n\n'))
print("Number of tabs", text.count('\t'))
print("Number of 2 tabs", text.count('\t\t'))
print("Number of tab and newline", text.count('\n\t'))

Initial text lenght: 159
Number of 2 newlines 4
Number of tabs 3
Number of 2 tabs 1
Number of tab and newline 1


In [ ]:
print('Manual approach:')
# replace every newline and every tab with a space
new_text = text.replace('\n', ' ').replace('\t', ' ')
print(new_text)
print("Number of newlines", new_text.count('\n'))
print("Number of tabs", new_text.count('\t'))
print("Number of 2 spaces", new_text.count('  '))
print('Manual text lenght', len(new_text))

Manual approach:
DEAR STUDENTS,  THE TEXT MINING LAB STARTS ON THE 10TH OF NOVEMBER 2026 IN U9.   PLEASE BRING YOUR LAPTOP, AND INSTALL PYTHON   BEST REGARDS,      SEE YOU SOON
Number of newlines 0
Number of tabs 0
Number of 2 spaces 6
Manual text lenght 159


### What does the manual approach do, and what does it miss?

Each `\n` and each `\t` becomes **one** space, so the words are never glued. But a sequence of whitespaces becomes a sequence of spaces:

| original text | after `.replace('\n', ' ').replace('\t', ' ')` |
|---|---|
| `STUDENTS,\n\nTHE` | `STUDENTS,  THE` (2 spaces) |
| `U9.\n\n\nPLEASE` | `U9.   PLEASE` (3 spaces) |
| `REGARDS,\n\n\t\t\t SEE` | `REGARDS,      SEE` (6 spaces) |

Adding `.replace('\n\n', ' ')` at the end does not help: at that point there are no `\n` left, they have already been replaced.
`replace` only works with a fixed string, so it cannot say "any number of whitespaces". A regex can: let's try it.

In [ ]:
print('\nRegex library (WRONG version, see below):')
import re
# \s\s matches 2 whitespaces at a time and replaces them with nothing
new_text2 = re.sub(r'\s\s', '', text)
print(new_text2)
print("Number of 2 newlines", new_text2.count('\n\n'))
print("Number of tabs", new_text2.count('\t'))
print("Number of 2 tabs", new_text2.count('\t\t'))
print("Number of tab and newline", new_text2.count('\n\t'))
print('Regex text lenght', len(new_text2))


Regex library (WRONG version, see below):
DEAR STUDENTS,THE TEXT MINING LAB STARTS ON THE 10TH OF NOVEMBER 2026 IN U9.
PLEASE BRING YOUR LAPTOP, AND INSTALL PYTHON
BEST REGARDS,SEE YOU SOON
Number of 2 newlines 0
Number of tabs 0
Number of 2 tabs 0
Number of tab and newline 0
Regex text lenght 147


### Why is `re.sub(r'\s\s', '', text)` wrong?

`\s\s` matches **exactly 2** whitespaces (spaces, `\n`, `\t`) and replaces them with **nothing** (`''`). This causes 2 problems, all visible in the output above:

| problem | original text | after `\s\s` → `''` |
|---|---|---|
| 2 whitespaces are removed → the words get glued | `STUDENTS,\n\nTHE` | `STUDENTS,THE` |
| 3 whitespaces: 2 are removed, 1 is left | `U9.\n\n\nPLEASE` | `U9.\nPLEASE` |


The glued words are a problem later: when we split the text into words, `STUDENTS,THE` becomes a single token.

The fix (as in the slides):
- `\s+` matches **1 or more** whitespaces, however many there are;
- we replace them with **one space** `' '`, not with nothing.

In [ ]:
# Correct version 1: regex
# \s+ = one or more whitespaces, replaced by a single space
clean_text = re.sub(r'\s+', ' ', text)

# Correct version 2: string methods
# split() cuts on any whitespace, " ".join() puts the words back with one space
# (it also removes the whitespaces at the beginning and at the end)
clean_text2 = " ".join(text.split())

print(clean_text)
print("Number of 2 newlines", clean_text.count('\n\n'))
print("Number of tabs", clean_text.count('\t'))
print('Regex text lenght', len(clean_text))
print('String methods text lenght', len(clean_text2))

text = clean_text

DEAR STUDENTS, THE TEXT MINING LAB STARTS ON THE 10TH OF NOVEMBER 2026 IN U9. PLEASE BRING YOUR LAPTOP, AND INSTALL PYTHON BEST REGARDS, SEE YOU SOON
Number of 2 newlines 0
Number of tabs 0
Regex text lenght 149
String methods text lenght 149


# Case folding



In [ ]:
print("Initial text:")
print(text)
lower = text.lower()

print()
print("Final text:")
print(lower)

Initial text:
DEAR STUDENTS, THE TEXT MINING LAB STARTS ON THE 10TH OF NOVEMBER 2026 IN U9. PLEASE BRING YOUR LAPTOP, AND INSTALL PYTHON BEST REGARDS, SEE YOU SOON

Final text:
dear students, the text mining lab starts on the 10th of november 2026 in u9. please bring your laptop, and install python best regards, see you soon


# Remove Links

We remove the links **before** the punctuation: otherwise `https://www.example.com` becomes `httpswwwexamplecom` and the regex no longer finds it.

In [ ]:
def remove_links(text):
  # Removing all the occurrences of links that start with http:// or https://
  text = re.sub(r'https?://\S+', ' ', text)

  # Remove all the occurrences of text that ends with .com
  text = re.sub(r"www\.[A-Za-z]*\.com", " ", text)
  return text

link_text = "For more information visit www.example.com or https://www.example.com."
print("Before:", link_text)
print("After: ", remove_links(link_text))

Before: For more information visit www.example.com or https://www.example.com.
After:  For more information visit   or  


These regexes only find the links that start with `http://`, `https://` or `www.`. Links written in other ways are not found, for example `mongodb.com` (no `www`):

In [ ]:
link_text = "Download mongoDB from mongodb.com"
print("Before:", link_text)
print("After: ", remove_links(link_text))

Before: Download mongoDB from mongodb.com
After:  Download mongoDB from mongodb.com


In [ ]:
# A better regex: domains with or without www, including an eventual port (e.g. .com:8080)
def remove_domains(text):
  return re.sub(r'(www\.)?[A-Za-z0-9_-]+\.[A-Za-z]{2,}(:\d+)?', ' ', text)

print("Before:", link_text)
print("After: ", remove_domains(link_text))

# Remove HTML Tags
This function utilizes **BeautifulSoup**, a powerful library for parsing HTML and XML documents in Python:

1. **`BeautifulSoup(text, "html.parser")`**: Parses the raw string and constructs a hierarchical document tree representing the HTML structure.
2. **`soup.get_text(separator=" ")`**: Traverses the document tree to extract all text content while discarding HTML tags (like `<IMG>...</IMG>`). The `separator=" "` parameter ensures that words separated by tags are joined with a space instead of being glued together.


In [ ]:
from bs4 import BeautifulSoup

sample = 'This is a nice place to live. <IMG>Image description</IMG>'

def strip_html_tags(text):
    # Initiating BeautifulSoup object soup.
    soup = BeautifulSoup(text, "html.parser")

    # Get all the text other than html tags.
    stripped_text = soup.get_text(separator=" ")
    return stripped_text

print("Before:", sample)
print('After: ', strip_html_tags(sample))

Before: This is a nice place to live. <IMG>Image description</IMG>
After:  This is a nice place to live.  Image description


# Remove punctuation and other Symbols

Every method to remove punctuation answers two questions:

1. **Which characters** are removed?
2. **What do we put in their place**: nothing `''` or a space `' '`?

Different answers give different results. We try two methods on the same examples and compare them.

In [ ]:
# Method 1: string.punctuation, a FIXED list of 32 ASCII symbols
import string
print("Library's punctuation: ", string.punctuation)

# The list does not contain '€', so we build a custom one
custom = list(string.punctuation)

custom.append('€')

print('Custom list:', custom,'\n')

# Defining the functions to remove punctuation: keep only the characters NOT in the list
def remove_punctuation(text):
    punctuationfree = "".join([i for i in text if i not in string.punctuation])
    return punctuationfree

def remove_custom_punctuation(text):
    punctuationfree = "".join([i for i in text if i not in custom])
    return punctuationfree

sm_text= 'Everything is ok! €€'
print("Simple Example:          ", sm_text)
print("With string.punctuation: ", remove_punctuation(sm_text))
print("With the custom list:    ", remove_custom_punctuation(sm_text))

Library's punctuation:  !"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
Custom list: ['!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', ':', ';', '<', '=', '>', '?', '@', '[', '\\', ']', '^', '_', '`', '{', '|', '}', '~', '€'] 

Simple Example:           Everything is ok! €€
With string.punctuation:  Everything is ok €€
With the custom list:     Everything is ok 


In [ ]:
# Method 2: regex
# [^\w\s] = any character that is NOT a letter/digit/underscore (\w) and NOT a whitespace (\s)
# We do not list the symbols to remove: we list the characters to KEEP
def remove_punctuation_regex(text, replace_with=''):
    return re.sub(r'[^\w\s]', replace_with, text)

print("Simple Example:", sm_text)
print("With the regex:", remove_punctuation_regex(sm_text))

Simple Example: Everything is ok! €€
With the regex: Everything is ok 


### Question 1: which characters are removed?

The two methods describe the punctuation in opposite ways:

| | Method 1: `string.punctuation` | Method 2: regex `[^\w\s]` |
|---|---|---|
| how it works | removes the characters **in a list** | removes everything that is **not** a letter, a digit, `_` or a whitespace |
| `€`, `£`, `°`, `’` (curly apostrophe) | **kept**: they are not in the list, we must add them by hand | **removed** automatically |
| `_` (underscore) | **removed**: it is in the list | **kept**: for the regex `_` is a "word" character (`\w`) |

The next cell shows these differences on some examples: `laptop,and` (as in the sample text), `(e.g. with anaconda.)`, `python 3.12` and `5k’s`.

In [ ]:
examples = [
    'Everything is ok! €€',
    'laptop,and install',      # from the sample text
    '(e.g. with anaconda.)',   # abbreviation
    'python 3.12',             # decimal number
    "don't",
    '5k’s',                    # curly apostrophe
    'snake_case',
    '30°C £5',
]

# quotes around each result, so the spaces are visible
show = lambda s: f"'{s}'"

comparison = pd.DataFrame({
    'original': [show(t) for t in examples],
    'string.punctuation → ""': [show(remove_punctuation(t)) for t in examples],
    'custom list → ""': [show(remove_custom_punctuation(t)) for t in examples],
    'regex → ""': [show(remove_punctuation_regex(t)) for t in examples],
    'regex → " "': [show(remove_punctuation_regex(t, ' ')) for t in examples],
})
display(comparison)

,original,"string.punctuation → """"","custom list → """"","regex → """"","regex → "" """
0,'Everything is ok! €€','Everything is ok €€','Everything is ok ','Everything is ok ','Everything is ok '
1,"'laptop,and install'",'laptopand install','laptopand install','laptopand install','laptop and install'
2,'(e.g. with anaconda.)','eg with anaconda','eg with anaconda','eg with anaconda',' e g with anaconda '
3,'python 3.12','python 312','python 312','python 312','python 3 12'
4,'don't','dont','dont','dont','don t'
5,'5k’s','5k’s','5k’s','5ks','5k s'
6,'snake_case','snakecase','snakecase','snake_case','snake_case'
7,'30°C £5','30°C £5','30°C £5','30C 5','30 C 5'


### Question 2: what do we put in their place?

Compare the last two columns of the table (same regex, different replacement):

| original | → `''` (nothing) | → `' '` (a space) |
|---|---|---|
| `laptop,and install` | `laptopand install` ❌ two words glued | `laptop and install` ✅ |
| `(e.g. with anaconda.)` | `eg with anaconda` ✅ one token | `' e g  with anaconda  '` ❌ `e`, `g` + extra spaces |
| `don't` | `dont` ✅ | `don t` ❌ two tokens |
| `5k’s` | `5ks` | `5k s` |

- `''` keeps together what belongs together (`e.g.`, `don't`), but **glues** words separated only by a symbol (`laptop,and`).
- `' '` keeps the words apart, but **splits** abbreviations and contractions, and leaves **extra spaces** (we must remove the whitespaces again).

**No method is right in every case**: we choose by looking at our data and at the downstream task.
In the SMS dataset many words are written with apostrophes (`don't`, `I'm`, `it's`): with `''` they stay one token, so below we use `''`.



In [ ]:
print("Spam Example")

# Storing the punctuation free text (method 1, replacing with nothing)
data_spam['text'] = data_spam['text'].apply(lambda x: remove_punctuation(x))
display(data_spam.head(2))

Spam Example


,label,text
0,ham,Go until jurong point crazy Available only in ...
1,ham,Ok lar Joking wif u oni


# Remove numbers

As for the punctuation, every method answers two questions:

1. **What do we remove**: only the digits, only the numbers standing alone, or the whole words that contain a digit (`2nd`, `4U`)?
2. **What is left behind**: extra spaces, pieces of words, symbols?

We try three regexes on the same examples and compare them.

In [ ]:
n_text = "DEAR STUDENTS, THE TEXT MINING LAB STARTS ON THE 10TH OF NOVEMBER 2026 IN U9. PLEASE BRING YOUR LAPTOP, AND INSTALL PYTHON BEST REGARDS, SEE YOU SOON"
print("Original:   ", n_text)

# Method 1: \d+ matches any sequence of one or more digits anywhere in the text.
# This will remove digits even if they are part of a word, leaving fragments (e.g., '10TH' -> 'TH').
print(r"\d+        :", re.sub(r'\d+', '', n_text))

# Method 2: \b\d+\b matches only standalone numbers.
# \b represents a word boundary. It matches transitions between a word character (letters/digits)
# and a non-word character (like spaces or punctuation). This keeps mixed words like '10TH' intact.
print(r"\b\d+\b    :", re.sub(r'\b\d+\b', '', n_text))

# Method 3: \w*\d\w* matches any whole word that contains at least one digit.
# \w* matches zero or more letters, digits, or underscores before and after the digit,
# thus removing the entire token containing the number (e.g., completely deleting '10TH').
print(r"\w*\d\w*   :", re.sub(r'\w*\d\w*', '', n_text))

Original:    DEAR STUDENTS, THE TEXT MINING LAB STARTS ON THE 10TH OF NOVEMBER 2026 IN U9. PLEASE BRING YOUR LAPTOP, AND INSTALL PYTHON BEST REGARDS, SEE YOU SOON
\d+        : DEAR STUDENTS, THE TEXT MINING LAB STARTS ON THE TH OF NOVEMBER  IN U. PLEASE BRING YOUR LAPTOP, AND INSTALL PYTHON BEST REGARDS, SEE YOU SOON
\b\d+\b    : DEAR STUDENTS, THE TEXT MINING LAB STARTS ON THE 10TH OF NOVEMBER  IN U9. PLEASE BRING YOUR LAPTOP, AND INSTALL PYTHON BEST REGARDS, SEE YOU SOON
\w*\d\w*   : DEAR STUDENTS, THE TEXT MINING LAB STARTS ON THE  OF NOVEMBER  IN . PLEASE BRING YOUR LAPTOP, AND INSTALL PYTHON BEST REGARDS, SEE YOU SOON


### Question 1: what do we remove?

| regex | what it matches | `the 10TH` or `U9` |
|---|---|---|
| `\d+` | every sequence of digits, **also inside a word** | `the TH`, `U`: fragments of words are left behind |
| `\b\d+\b` | only a number **standing alone** (`\b` = word boundary) | `the 10TH`, `U9`: `10TH` and `U9` are not standalone numbers, so they stay intact |
| `\w*\d\w*` | the **whole word** that contains at least one digit (`\w*` = letters before and after) | `the `, ` `: the entire token containing the digit is removed |

The three methods give the same result on numbers standing alone (`2026`). They differ only on the **words that mix letters and digits**: these are very common in SMS (`2day`, `B4`, `4U`).

The next cell compares them on more examples: the first row comes from the SMS dataset (row 2), and the `4U`/`2day`/`B4` ones are SMS-style words.

In [ ]:
examples = [
    'Free entry in 2 a wkly comp',
    'July 1776, during',
    'the 3rd of march 2026',
    'in room u14',
    'call 4U 2day',
    'B4 tomorrow',
    'python 3.12',
]

# quotes around each result, so the spaces are visible
show = lambda s: f"'{s}'"

comparison = pd.DataFrame({
    'original': [show(t) for t in examples],
    '\\d+': [show(re.sub(r'\d+', '', t)) for t in examples],
    '\\b\\d+\\b': [show(re.sub(r'\b\d+\b', '', t)) for t in examples],
    '\\w*\\d\\w*': [show(re.sub(r'\w*\d\w*', '', t)) for t in examples],
})
display(comparison)

,original,\d+,\b\d+\b,\w*\d\w*
0,'Free entry in 2 a wkly comp','Free entry in a wkly comp','Free entry in a wkly comp','Free entry in a wkly comp'
1,"'July 1776, during'","'July , during'","'July , during'","'July , during'"
2,'the 3rd of march 2026','the rd of march ','the 3rd of march ','the of march '
3,'in room u14','in room u','in room u14','in room '
4,'call 4U 2day','call U day','call 4U 2day','call '
5,'B4 tomorrow','B tomorrow','B4 tomorrow',' tomorrow'
6,'python 3.12','python .','python .','python .'


### Question 2: what is left behind?

Look at the spaces and the symbols in the table:

| original | after `\d+` → `''` | what is left |
|---|---|---|
| `Free entry in 2 a wkly comp` | `Free entry in  a wkly comp` | **2 spaces**: the number is gone, the spaces around it stay |
| `July 1776, during` | `July , during` | a space **before the comma** |
| `python 3.12` | `python .` | the **dot** of the decimal number |
| `call 4U 2day` | `call U day` | **pieces of words**: `U`, `day` |

- The extra spaces are easy to fix: we collapse the whitespaces again, as in the first section (`re.sub(r'\s+', ' ', text)`).
- The leftover symbols depend on the **order of the steps**: in this notebook the punctuation is removed **before** the numbers, so `3.12` has already become `312` and nothing is left. If we remove the numbers first, the `.` stays.

**Which method for the SMS?** It depends on the downstream task. Words like `2day` or `4U` are typical of spam messages, so they can be useful to recognise them: `\b\d+\b` keeps them, `\d+` cuts them into pieces, `\w*\d\w*` removes them. Below we use `\d+`.

> **Note on the slides:** the slides remove punctuation and digits in a single step, `re.sub(r"[^\w\s]|\d", "", text)`, which is the same as `\d+` here: `3rd` → `rd`, `u14` → `u`.

In [ ]:
print("Spam Example")

# Removing the digits from the SMS (method 1)
data_spam['text']= data_spam['text'].apply(lambda x: re.sub(r'\d+', '', x))
display(data_spam.head(5))

Spam Example


,label,text
0,ham,Go until jurong point crazy Available only in ...
1,ham,Ok lar Joking wif u oni
2,spam,Free entry in a wkly comp to win FA Cup final...
3,ham,U dun say so early hor U c already then say
4,ham,Nah I dont think he goes to usf he lives aroun...


# Remove repeated characters

Two questions: **which characters** do we shorten, and **how many repetitions** do we keep?

In [ ]:
def character_repetition(text):
    # Pattern matching for all alphabetic characters where any single character
    # repeats three or more times consecutively.
    # ([A-Za-z])  - Capturing group. Captures any alphabetic character.
    # \1{2,}      - Matches the captured character repeating at least two more
    # times after the initial match.
    # For example, this pattern matches 'aaa', 'BBBB', etc.
    pattern_alpha = re.compile(r"([A-Za-z])\1{2,}", re.DOTALL)

    # Limiting all the repetition to two characters.
    formatted_text = pattern_alpha.sub(r"\1\1", text)

    # Pattern matching for all the punctuations that can occur
    pattern_punct = re.compile(r'([!$?])\1{1,}')

    # Limiting punctuations in previously formatted string to only one.
    combined_formatted = pattern_punct.sub(r"\1", formatted_text)
    return combined_formatted

repeated = "Realllllllllyyyyy good!!!!!"
print("Before:", repeated)
print("After: ", character_repetition(repeated))

Before: Realllllllllyyyyy good!!!!!
After:  Reallyy good!


In [ ]:
# Slides version: (.) = ANY character (letters, digits, punctuation, spaces)
def character_repetition_slides(text):
    return re.sub(r"(.)\1{2,}", r"\1\1", text)

examples = ['sooo good', 'Realllllllllyyyyy good!!!!!', 'WHAT A GAME!!', 'Ok lar...', 'win 1000 cash']

# quotes around each result, so the spaces are visible
show = lambda s: f"'{s}'"

display(pd.DataFrame({
    'original': [show(t) for t in examples],
    'notebook': [show(character_repetition(t)) for t in examples],
    'slides (.)\\1{2,}': [show(character_repetition_slides(t)) for t in examples],
}))

,original,notebook,"slides (.)\1{2,}"
0,'sooo good','soo good','soo good'
1,'Realllllllllyyyyy good!!!!!','Reallyy good!','Reallyy good!!'
2,'WHAT A GAME!!','WHAT A GAME!','WHAT A GAME!!'
3,'Ok lar...','Ok lar...','Ok lar..'
4,'win 1000 cash','win 1000 cash','win 100 cash'


- **Which characters:** the notebook shortens only letters `[A-Za-z]` and `! $ ?`. The slides shorten **any** character, also digits: `1000` → `100` changes the number!
- **How many to keep:** letters are cut to **2**, because we cannot know if the word has a double letter (`good`, `really`). Punctuation is cut to 1 in the notebook, to 2 in the slides (`!!!!!` → `!!`).

# Spelling Correction

In [ ]:
# Import the TextBlob class from the textblob library
from textblob import TextBlob

wrong = "raeyly good!!!!!"

def correct_me(text):
  # Initialize a TextBlob object with the input text
  text_blb = TextBlob(text)

  # Call the .correct() method, which checks spelling word by word
  text_corrected = text_blb.correct()

  return text_corrected

print("Before:", wrong)
print("After: ", correct_me(wrong))

Before: raeyly good!!!!!
After:  really good!!!!!


In [ ]:
# Tricky example:
tricky_example = "sooo good"

print("Before:", tricky_example)
print("After: ", correct_me(tricky_example))

Before: sooo good
After:  soon good


It corrected to the wrong word: `sooo` → `soon`! We need to combine it with the regex to remove multiple repetitions first: `sooo` → `soo` → `so`.

In [ ]:
print("Before:", tricky_example)
print("After repetition removal:", character_repetition(tricky_example))
print("After repetition removal + correction:", correct_me(character_repetition(tricky_example)))

Before: sooo good
After repetition removal: soo good
After repetition removal + correction: so good


# Removing Emoji

In [ ]:
!pip install emoji
!pip install demoji

In [ ]:
import emoji
import demoji

# Print an emoji using its Unicode code.
# \U0001f605 represents the "grinning face with sweat" emoji.
# The double backslash (\\) prints the Unicode code literally.
print("\U0001f605 is  \\U0001f605 in utf")


# Print the emoji together with its textual name
# used by the emoji library.
print('\U0001f605 is :grinning_face_with_sweat: in emoji library')


# Convert the textual emoji description into its
# corresponding graphical emoji using emoji.emojize().
print(emoji.emojize('emojized :grinning_face_with_sweat:'))

😅 is  \U0001f605 in utf
😅 is :grinning_face_with_sweat: in emoji library
emojized 😅


In [ ]:
print("\n Working with Text Messages")

tweet = """\
... Had a great day today! ☀️
... Just enjoying a coffee with my friends ☕❤️
... Life is good! 😊😊"""

print(tweet)

# Find and display all emojis contained in the text.
display(demoji.findall(tweet))


 Working with Text Messages
... Had a great day today! ☀️
... Just enjoying a coffee with my friends ☕❤️
... Life is good! 😊😊


{'☕': 'hot beverage',
 '😊': 'smiling face with smiling eyes',
 '❤️': 'red heart',
 '☀️': 'sun'}

In [ ]:
# Remove the emojis: the information they carry is lost
print(emoji.replace_emoji(tweet, ""))

print()

# Turn the emojis into words: the information stays (e.g. :fire: can help for sentiment)
print(emoji.demojize(tweet))

... Had a great day today! 
... Just enjoying a coffee with my friends 
... Life is good! 

... Had a great day today! :sun:
... Just enjoying a coffee with my friends :hot_beverage::red_heart:
... Life is good! :smiling_face_with_smiling_eyes::smiling_face_with_smiling_eyes:


# Tokenization

Tokenization is the process of breaking down a piece of text (such as a sentence or paragraph) into smaller units called tokens. These tokens can be words, phrases, individual characters, or word pieces known as sub-words, depending on the purpose and type of tokenization.

In [ ]:
import re

# Define the text that we want to tokenize.
tokenize = text

print('Before:            ', tokenize)


# Split the text using Python's built-in split() method.
# By default, split() separates the text wherever there is whitespace
# (spaces, tabs, newlines).
print('String Operator:   ', tokenize.split())


# Define a custom tokenization function using Regular Expressions.
def tokenization(text):

    # Split the text whenever one or more non-word characters are found.
    # \W matches characters that are not letters, digits or underscores.
    # + means one or more consecutive non-word characters.
    tokens = re.split(r'\W+', text)

    return tokens


print("Simple Regex:      ", tokenization(tokenize))


# This tokenizer separates words and punctuation into individual tokens.
from nltk.tokenize import WordPunctTokenizer

# Create a WordPunctTokenizer object and use it to tokenize the text.
new_text4 = WordPunctTokenizer().tokenize(tokenize)

print("WordPunctTokenizer:", new_text4)

Before:             DEAR STUDENTS, THE TEXT MINING LAB STARTS ON THE 10TH OF NOVEMBER 2026 IN U9. PLEASE BRING YOUR LAPTOP, AND INSTALL PYTHON BEST REGARDS, SEE YOU SOON
String Operator:    ['DEAR', 'STUDENTS,', 'THE', 'TEXT', 'MINING', 'LAB', 'STARTS', 'ON', 'THE', '10TH', 'OF', 'NOVEMBER', '2026', 'IN', 'U9.', 'PLEASE', 'BRING', 'YOUR', 'LAPTOP,', 'AND', 'INSTALL', 'PYTHON', 'BEST', 'REGARDS,', 'SEE', 'YOU', 'SOON']
Simple Regex:       ['DEAR', 'STUDENTS', 'THE', 'TEXT', 'MINING', 'LAB', 'STARTS', 'ON', 'THE', '10TH', 'OF', 'NOVEMBER', '2026', 'IN', 'U9', 'PLEASE', 'BRING', 'YOUR', 'LAPTOP', 'AND', 'INSTALL', 'PYTHON', 'BEST', 'REGARDS', 'SEE', 'YOU', 'SOON']
WordPunctTokenizer: ['DEAR', 'STUDENTS', ',', 'THE', 'TEXT', 'MINING', 'LAB', 'STARTS', 'ON', 'THE', '10TH', 'OF', 'NOVEMBER', '2026', 'IN', 'U9', '.', 'PLEASE', 'BRING', 'YOUR', 'LAPTOP', ',', 'AND', 'INSTALL', 'PYTHON', 'BEST', 'REGARDS', ',', 'SEE', 'YOU', 'SOON']


What changes between the three methods?
- `split()` cuts only on the spaces: the punctuation stays attached to the word (`STUDENTS,`, `LAPTOP,`).
- The regex `\W+` cuts on everything that is not a letter or a digit: the punctuation is lost.
- `WordPunctTokenizer` separates words from punctuation, keeping punctuation as independent tokens. However, consecutive punctuation marks can be grouped together into a single token (for example, `.`).

In [ ]:
print('\nWorking with Tweets\n')
### TweetTokenizer: specifically designed to handle tweets
from nltk.tokenize import TweetTokenizer

# Create a TweetTokenizer object.
tknzr = TweetTokenizer()

# Define a sample tweet containing:
# - Repeated letters (cooool)
# - A hashtag (#dummysmiley)
# - Emoticons (:-), :-P, <3)
# - Special characters and arrows (<, >, ->, <--)
tweet_tok = "This is a cooool #dummysmiley: :-) :-P <3 and some arrows < > -> <--"
print('TweetTokenizer:', tknzr.tokenize(tweet_tok))
print("Simple Regex:  ", tokenization(tweet_tok))

# Example from the slides: the regex loses @, the smiley and #
tweet_slides = "@unimib Text mining is so cool :-) #NLP"
print('\nTweetTokenizer:', tknzr.tokenize(tweet_slides))
print("Simple Regex:  ", tokenization(tweet_slides))


Working with Tweets

TweetTokenizer: ['This', 'is', 'a', 'cooool', '#dummysmiley', ':', ':-)', ':-P', '<3', 'and', 'some', 'arrows', '<', '>', '->', '<--']
Simple Regex:   ['This', 'is', 'a', 'cooool', 'dummysmiley', 'P', '3', 'and', 'some', 'arrows', '']

TweetTokenizer: ['@unimib', 'Text', 'mining', 'is', 'so', 'cool', ':-)', '#NLP']
Simple Regex:   ['', 'unimib', 'Text', 'mining', 'is', 'so', 'cool', 'NLP']


# Sentence split

Sentence tokenization is useful in natural language processing (NLP) for a variety of tasks where understanding, analyzing, or processing text at the sentence level is important.


In [ ]:
from nltk.tokenize import sent_tokenize, word_tokenize
new_text3 = sent_tokenize(text)
display(new_text3)

['DEAR STUDENTS, THE TEXT MINING LAB STARTS ON THE 10TH OF NOVEMBER 2026 IN U9.',
 'PLEASE BRING YOUR LAPTOP, AND INSTALL PYTHON BEST REGARDS, SEE YOU SOON']

# Remove stop words

Removing stop words is a common preprocessing step in natural language processing (NLP) and text analysis. Stop words are commonly used words in a language (such as "the," "is," "in," "and") that typically **carry little semantic weight and do not add much to the meaning** of a sentence.
Removing them reduces computational costs, and improves model performance by eliminating words that add little value to analysis.

However, **there are cases where it may not be ideal**.
For very short texts, like tweets, stop words might carry critical meaning.
In question-answering systems or sentiment analysis, stop words like “not” or “but” can influence sentiment and should not be removed.

In [ ]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [ ]:
from nltk.tokenize import WordPunctTokenizer

# Load the list of English stopwords from the NLTK corpus.
stop_words = nltk.corpus.stopwords.words('english')

def remove_stopwords(text):
  # Split the text into individual tokens using WordPunctTokenizer.
  tokenized_text = WordPunctTokenizer().tokenize(text)

  # Keep only the words that are NOT included in the stopwords list.
  # w.lower() converts each token to lowercase before checking,
  # so that stopwords are removed regardless of capitalization.
  tokenized_text_without_stopwords = [
      w for w in tokenized_text if not w.lower() in stop_words]

  return tokenized_text_without_stopwords

stop_me = "We are ready to travel to NY."
print('With Stopwords:   ', (stop_me.split()))
print('Without Stopwords:', remove_stopwords(stop_me))

# Remove stopwords from all the SMS
print('SMS: Total tokens with stopwords:   ', len(WordPunctTokenizer().tokenize(raw)))
clean_re = remove_stopwords(raw)
print('SMS: Total tokens without stopwords:', len(clean_re))

With Stopwords:    ['We', 'are', 'ready', 'to', 'travel', 'to', 'NY.']
Without Stopwords: ['ready', 'travel', 'NY', '.']
SMS: Total tokens with stopwords:    109366
SMS: Total tokens without stopwords: 72484


# Stemming

Stemming is a text preprocessing technique used in natural language processing (NLP) to reduce words to their root or base form. This root form is called the stem. Stemming algorithms typically remove common word endings, such as -ing, -ed, -s, and -ly, to bring words with similar meanings down to a shared base form.

In [ ]:
from nltk.stem.porter import PorterStemmer

# Defining the object for stemming
porter_stemmer = PorterStemmer()

def stemming(text):
  # Apply the PorterStemmer to each word in the list.
  stem_text = [porter_stemmer.stem(word) for word in text]
  return stem_text

# Apply the remove_stopwords() function to every SMS in the 'text' column.
# The resulting lists of tokens are stored in a new column called 'no_stopwords'.
data_spam['no_stopwords'] = data_spam['text'].apply(lambda x: remove_stopwords(x))


# Apply stemming to the 'no_stopwords' column.
# Each SMS is transformed into a list of stemmed tokens.
# The results are stored in a new column called 'msg_stemmed'.
data_spam['msg_stemmed'] = data_spam['no_stopwords'].apply(lambda x: stemming(x))
display(data_spam.head(5))

,label,text,no_stopwords,msg_stemmed
0,ham,Go until jurong point crazy Available only in ...,"[Go, jurong, point, crazy, Available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre..."
1,ham,Ok lar Joking wif u oni,"[Ok, lar, Joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]"
2,spam,Free entry in a wkly comp to win FA Cup final...,"[Free, entry, wkly, comp, win, FA, Cup, final,...","[free, entri, wkli, comp, win, fa, cup, final,..."
3,ham,U dun say so early hor U c already then say,"[U, dun, say, early, hor, U, c, already, say]","[u, dun, say, earli, hor, u, c, alreadi, say]"
4,ham,Nah I dont think he goes to usf he lives aroun...,"[Nah, dont, think, goes, usf, lives, around, t...","[nah, dont, think, goe, usf, live, around, tho..."


# Lemmatization

Lemmatization is a more sophisticated process that reduces words to their lemma, which is the dictionary or canonical form of a word. It involves considering the context and part of speech (POS) of each word to ensure accurate transformation.

In [ ]:
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
from nltk.stem import WordNetLemmatizer

# Defining the object for Lemmatization
wordnet_lemmatizer = WordNetLemmatizer()

# Defining the function for lemmatization
def lemmatizer(text):
  # Apply the lemmatizer to each word in the list.
  lemm_text = [wordnet_lemmatizer.lemmatize(word) for word in text]
  return lemm_text

# Apply lemmatization to the 'no_stopwords' column.
# Each SMS is transformed into a list of lemmatized tokens.
# The results are stored in a new column called 'msg_lemmatized'.
data_spam['msg_lemmatized'] = data_spam['no_stopwords'].apply(lambda x: lemmatizer(x))
display(data_spam.head())

,label,text,no_stopwords,msg_stemmed,msg_lemmatized
0,ham,Go until jurong point crazy Available only in ...,"[Go, jurong, point, crazy, Available, bugis, n...","[go, jurong, point, crazi, avail, bugi, n, gre...","[Go, jurong, point, crazy, Available, bugis, n..."
1,ham,Ok lar Joking wif u oni,"[Ok, lar, Joking, wif, u, oni]","[ok, lar, joke, wif, u, oni]","[Ok, lar, Joking, wif, u, oni]"
2,spam,Free entry in a wkly comp to win FA Cup final...,"[Free, entry, wkly, comp, win, FA, Cup, final,...","[free, entri, wkli, comp, win, fa, cup, final,...","[Free, entry, wkly, comp, win, FA, Cup, final,..."
3,ham,U dun say so early hor U c already then say,"[U, dun, say, early, hor, U, c, already, say]","[u, dun, say, earli, hor, u, c, alreadi, say]","[U, dun, say, early, hor, U, c, already, say]"
4,ham,Nah I dont think he goes to usf he lives aroun...,"[Nah, dont, think, goes, usf, lives, around, t...","[nah, dont, think, goe, usf, live, around, tho...","[Nah, dont, think, go, usf, life, around, though]"


In [ ]:
# Examples from the slides: stemming vs lemmatization
words = ['fish', 'fishes', 'fishing', 'fished', 'argue', 'argued', 'arguing', 'thought', 'thinking', 'thinks']

display(pd.DataFrame({
    'word': words,
    # rules: cuts the suffixes, the result may not be a real word
    'stemmer': [porter_stemmer.stem(w) for w in words],
    # dictionary: without pos, every word is treated as a noun
    'lemmatizer': [wordnet_lemmatizer.lemmatize(w) for w in words],
    # dictionary: with pos='v', every word is treated as a verb
    "lemmatizer pos='v'": [wordnet_lemmatizer.lemmatize(w, pos='v') for w in words],
}))

,word,stemmer,lemmatizer,lemmatizer pos='v'
0,fish,fish,fish,fish
1,fishes,fish,fish,fish
2,fishing,fish,fishing,fish
3,fished,fish,fished,fish
4,argue,argu,argue,argue
5,argued,argu,argued,argue
6,arguing,argu,arguing,argue
7,thought,thought,thought,think
8,thinking,think,thinking,think
9,thinks,think,think,think


# Part-of-speech Tagging

Part-of-Speech (POS) Tagging is a process in natural language processing (NLP) that involves assigning grammatical tags, or parts of speech, to each word in a sentence. These tags represent the word's syntactic role, such as noun, verb, adjective, adverb, etc.

In [ ]:
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger_eng')

from textblob import TextBlob

tag = [
    "Bob and Carl are my good friends. " ,
    "Bob is getting married next year. " ,
    "Marriage is a big step in one’s life." ,
    "But friendship is a sacred bond between people." ,
]

for el in tag:
  result = TextBlob(el)
  print('\n', el)
  print(result.tags)

# NNP: Proper noun, identifies the names of specific people (Alice, Bob, Carl).
# CC: Coordinating conjunction, connects words or phrases (and, but).
# VBP: Present-tense verb used with plural subjects (are).
# VBZ: Present-tense verb used with third-person singular subjects (is).
# PRP$: Possessive pronoun, indicates ownership (my).
# JJ: Adjective, describes a noun (good, big, sacred).
# NNS: Plural noun, identifies more than one person or thing (friends, people).
# VBG: Verb in the gerund or present participle form (getting).
# VBN: Verb in the past participle form (married).
# NN: Singular common noun, identifies a person, place, thing or concept (marriage, year, life).
# DT: Determiner, introduces a noun (a).
# IN: Preposition, expresses relationships between words (in, between).
# CD: Cardinal number, identifies a numerical value (one).


 Bob and Carl are my good friends. 
[('Bob', 'NNP'), ('and', 'CC'), ('Carl', 'NNP'), ('are', 'VBP'), ('my', 'PRP$'), ('good', 'JJ'), ('friends', 'NNS')]

 Bob is getting married next year. 
[('Bob', 'NNP'), ('is', 'VBZ'), ('getting', 'VBG'), ('married', 'VBN'), ('next', 'JJ'), ('year', 'NN')]

 Marriage is a big step in one’s life.
[('Marriage', 'NN'), ('is', 'VBZ'), ('a', 'DT'), ('big', 'JJ'), ('step', 'NN'), ('in', 'IN'), ('one', 'CD'), ('’', 'NN'), ('s', 'JJ'), ('life', 'NN')]

 But friendship is a sacred bond between people.
[('But', 'CC'), ('friendship', 'NN'), ('is', 'VBZ'), ('a', 'DT'), ('sacred', 'JJ'), ('bond', 'NN'), ('between', 'IN'), ('people', 'NNS')]


[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


## Part-of-speech Tagging with SpaCy



In [ ]:
!pip install spacy
!python -m spacy download it_core_news_sm

import spacy

nlp = spacy.load("it_core_news_sm")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 24.6 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('it_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
tag_ita = [
    "Bob e Carl sono i miei buoni amici.",
    "Bob si sposerà l'anno prossimo.",
    "Il matrimonio è un grande passo nella vita di una persona.",
    "Ma l'amicizia è un legame sacro tra le persone.",
    "La tecnologia sta cambiando il mondo.",
    "Gli studenti studiano per superare gli esami.",
    "Il cambiamento climatico rappresenta un problema globale.",
    "I bambini giocano felicemente nel parco."
]

for el in tag_ita:
    result = nlp(el)
    print("\n", el)

    for token in result:
        print((token.text, token.pos_))


# PROPN: Proper noun, identifies specific names of people (Bob, Carl).
# CCONJ: Coordinating conjunction, connects words or clauses (e, ma).
# AUX: Auxiliary verb, helps form verb tenses (sono, è).
# DET: Determiner, introduces or specifies a noun (il, i, un, le).
# ADJ: Adjective, describes a noun (buoni, grande, prossimo, sacro).
# NOUN: Common noun, identifies people, objects or concepts (amici, matrimonio, vita).
# VERB: Verb, expresses an action or state (sposerà).
# PRON: Pronoun, replaces or refers to a noun (si).
# ADP: Adposition, expresses relationships between words (nella, di, tra).
# PUNCT: Punctuation mark, separates sentences or clauses.


 Bob e Carl sono i miei buoni amici.
('Bob', 'PROPN')
('e', 'CCONJ')
('Carl', 'PROPN')
('sono', 'AUX')
('i', 'DET')
('miei', 'DET')
('buoni', 'ADJ')
('amici', 'NOUN')
('.', 'PUNCT')

 Bob si sposerà l'anno prossimo.
('Bob', 'PROPN')
('si', 'PRON')
('sposerà', 'VERB')
("l'", 'DET')
('anno', 'NOUN')
('prossimo', 'ADJ')
('.', 'PUNCT')

 Il matrimonio è un grande passo nella vita di una persona.
('Il', 'DET')
('matrimonio', 'NOUN')
('è', 'AUX')
('un', 'DET')
('grande', 'ADJ')
('passo', 'NOUN')
('nella', 'ADP')
('vita', 'NOUN')
('di', 'ADP')
('una', 'DET')
('persona', 'NOUN')
('.', 'PUNCT')

 Ma l'amicizia è un legame sacro tra le persone.
('Ma', 'CCONJ')
("l'", 'DET')
('amicizia', 'NOUN')
('è', 'AUX')
('un', 'DET')
('legame', 'NOUN')
('sacro', 'ADJ')
('tra', 'ADP')
('le', 'DET')
('persone', 'NOUN')
('.', 'PUNCT')

 La tecnologia sta cambiando il mondo.
('La', 'DET')
('tecnologia', 'NOUN')
('sta', 'AUX')
('cambiando', 'VERB')
('il', 'DET')
('mondo', 'NOUN')
('.', 'PUNCT')

 Gli studenti st

# Chunking

Chunking in Natural Language Processing (NLP) refers to the process of grouping words (or tokens) into chunks, which are meaningful phrases or units based on syntactic structure. This is typically done after part-of-speech (POS) tagging and is often used to identify and extract phrases such as noun phrases (NP), verb phrases (VP), or prepositional phrases (PP).

In [ ]:
input_str = "A black television and a white stove were bought for the new apartment of John."
result = TextBlob(input_str)
print(result.tags)

# DT determiner
# JJ adjective ‘big’
# NN a noun

[('A', 'DT'), ('black', 'JJ'), ('television', 'NN'), ('and', 'CC'), ('a', 'DT'), ('white', 'JJ'), ('stove', 'NN'), ('were', 'VBD'), ('bought', 'VBN'), ('for', 'IN'), ('the', 'DT'), ('new', 'JJ'), ('apartment', 'NN'), ('of', 'IN'), ('John', 'NNP')]


In [ ]:
reg_exp = "NP: {<DT>*<JJ>*<NN>}"
rp = nltk.RegexpParser(reg_exp)
result = rp.parse(result.tags)

for el in result:
  print(el)
# print(result)

(NP A/DT black/JJ television/NN)
('and', 'CC')
(NP a/DT white/JJ stove/NN)
('were', 'VBD')
('bought', 'VBN')
('for', 'IN')
(NP the/DT new/JJ apartment/NN)
('of', 'IN')
('John', 'NNP')


In [ ]:
import nltk
from textblob import TextBlob

input_str = "A black television and a white stove were bought for the new apartment of John."

# POS TAGGING
result = TextBlob(input_str)

print("ORIGINAL SENTENCE:")
print(input_str)

print("\nPOS TAGGING:")
for word, tag in result.tags:
    print(f"{word:15} -> {tag}")

# CHUNKING: define grammatical patterns

reg_exp = r"""
    NP: {<DT>?<JJ>*<NN|NNS|NNP>+}
    VP: {<MD>?<VB.*><RB.*>*<VB.*>*}
    PP: {<IN><NP>}
"""

# Create the Chunk Parser
rp = nltk.RegexpParser(reg_exp)

# Apply Chunking
chunk_result = rp.parse(result.tags)

print("\nCHUNKING RESULT:")
print(chunk_result)

# Extract and display NP, VP and PP

print("\nIDENTIFIED PHRASES:")

for subtree in chunk_result.subtrees():
    if subtree.label() in ["NP", "VP", "PP"]:
        phrase = " ".join(word for word, tag in subtree.leaves())
        print(f"{subtree.label()}: {phrase}")

ORIGINAL SENTENCE:
A black television and a white stove were bought for the new apartment of John.

POS TAGGING:
A               -> DT
black           -> JJ
television      -> NN
and             -> CC
a               -> DT
white           -> JJ
stove           -> NN
were            -> VBD
bought          -> VBN
for             -> IN
the             -> DT
new             -> JJ
apartment       -> NN
of              -> IN
John            -> NNP

CHUNKING RESULT:
(S
  (NP A/DT black/JJ television/NN)
  and/CC
  (NP a/DT white/JJ stove/NN)
  (VP were/VBD bought/VBN)
  (PP for/IN (NP the/DT new/JJ apartment/NN))
  (PP of/IN (NP John/NNP)))

IDENTIFIED PHRASES:
NP: A black television
NP: a white stove
VP: were bought
PP: for the new apartment
NP: the new apartment
PP: of John
NP: John


# Entity Recognition

Named Entity Recognition (NER) is a fundamental task in Natural Language Processing (NLP) that involves identifying and classifying named entities in text into predefined categories. Named entities typically include specific items such as people, organizations, locations, dates, numerical values, and more. The goal of NER is to extract useful structured information from unstructured text. Common entity types include ORGANIZATION, PERSON, LOCATION, DATE, TIME, MONEY, and GPE (geo-political entity).

In [ ]:
nltk.download('maxent_ne_chunker')
nltk.download('words')

[nltk_data] Downloading package maxent_ne_chunker to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping chunkers/maxent_ne_chunker.zip.
[nltk_data] Downloading package words to /root/nltk_data...
[nltk_data]   Unzipping corpora/words.zip.


True

In [ ]:
import nltk
nltk.download('maxent_ne_chunker_tab')

from nltk import word_tokenize, pos_tag, ne_chunk

input_str = "Bill works for Apple so he went to Boston for a conference"
for el in ne_chunk(pos_tag(word_tokenize(input_str))):
  print(el)

# NLTK does not find the dates and uses GPE for the cities: for DATE we need a library like spaCy
input_slides = "Charles visited London on 25 December"
print()
for el in ne_chunk(pos_tag(word_tokenize(input_slides))):
  print(el)

# PERSON: Identifies names of people, such as Bill and Charles.
# ORG: Identifies organizations, companies or institutions, such as Apple and Google.
# GPE: Identifies geopolitical entities, such as cities, countries and states (Boston, London).
# LOC: Identifies geographical locations that are not geopolitical entities, such as mountains and rivers.
# DATE: Identifies dates or periods of time, such as 25 December or next year.
# TIME: Identifies specific times or time periods, such as 10:30 AM or three hours.
# MONEY: Identifies monetary values, such as $100 or 50 euros.
# PERCENT: Identifies percentages, such as 25% or 10 percent.
# FAC: Identifies buildings and facilities, such as airports, bridges and stadiums.
# PRODUCT: Identifies products, objects or commercial items.
# EVENT: Identifies named events, such as festivals, conferences and sporting competitions.
# NORP: Identifies nationalities, religious or political groups.
# LAW: Identifies named laws or legal documents.
# LANGUAGE: Identifies languages, such as English, Italian or French.

[nltk_data] Downloading package maxent_ne_chunker_tab to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package maxent_ne_chunker_tab is already up-to-date!


(PERSON Bill/NNP)
('works', 'VBZ')
('for', 'IN')
('Apple', 'NNP')
('so', 'IN')
('he', 'PRP')
('went', 'VBD')
('to', 'TO')
(GPE Boston/NNP)
('for', 'IN')
('a', 'DT')
('conference', 'NN')

(PERSON Charles/NNP)
('visited', 'VBD')
(GPE London/NNP)
('on', 'IN')
('25', 'CD')
('December', 'NNP')


## Entity Recognition with spaCy

spaCy uses a trained model that knows more entity types than NLTK, including `DATE`, `TIME`, `MONEY`.

In [ ]:
# !python -m spacy download en_core_web_sm   ###< run once, if the model is missing
import spacy

nlp = spacy.load("en_core_web_sm")   # small English model

for sentence in [input_str, input_slides]:
  doc = nlp(sentence)
  print(sentence)
  print([(ent.text, ent.label_) for ent in doc.ents])
  print()

Bill works for Apple so he went to Boston for a conference
[('Bill', 'PERSON'), ('Apple', 'ORG'), ('Boston', 'GPE')]

Charles visited London on 25 December
[('Charles', 'PERSON'), ('London', 'GPE'), ('25 December', 'DATE')]



# Putting it all together

Until now we applied each step on its own. A real pipeline applies them **in order**, and the order matters (e.g. links before punctuation).

We leave out spelling correction, stop words, stemming and lemmatization: whether to use them depends on the downstream task (and spelling correction is very slow on 5,572 SMS).

In [ ]:
def preprocess(text):
    text = text.lower()                 # case folding
    text = remove_links(text)           # links BEFORE punctuation
    text = strip_html_tags(text)        # HTML tags and entities (&lt; → <)
    text = remove_punctuation(text)
    text = re.sub(r'\d+', '', text)     # numbers
    text = character_repetition(text)
    text = re.sub(r'\s+', ' ', text).strip()   # at the end: the previous steps leave extra spaces
    return text

# Apply it to the original SMS (before the cleaning done in the sections above)
sms_clean = sms_original.apply(preprocess)

display(pd.DataFrame({'label': data_spam['label'], 'before': sms_original, 'after': sms_clean}).head(5))

,label,before,after
0,ham,"Go until jurong point, crazy.. Available only ...",go until jurong point crazy available only in ...
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif u oni
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,free entry in a wkly comp to win fa cup final ...
3,ham,U dun say so early hor... U c already then say...,u dun say so early hor u c already then say
4,ham,"Nah I don't think he goes to usf, he lives aro...",nah i dont think he goes to usf he lives aroun...


# TIME for Exercise 1.1: the typical words of spam and ham

Use `sms_original` (the SMS before any cleaning) and the `label` column of `data_spam`.

1. Apply `preprocess()` to the SMS, then `remove_stopwords()`.
2. Print the 15 most frequent words of the **spam** and of the **ham** messages.
3. Skip one step of `preprocess()` at a time (e.g. the numbers or the repetitions): which step changes the spam top 15 the most? Which new words enter the list?

```
# You can use:
from collections import Counter
Counter(list_of_words).most_common(15)
```

In [ ]:
# add your code here

### Solution 1.1

In [ ]:
from collections import Counter

# The steps of preprocess(), in the same order, so we can skip one at a time
steps = {
    'lower':       lambda t: t.lower(),
    'links':       remove_links,
    'html':        strip_html_tags,
    'punctuation': remove_punctuation,
    'numbers':     lambda t: re.sub(r'\d+', '', t),
    'repetitions': character_repetition,
}

def run_pipeline(text, skip=None):
    for name, step in steps.items():
        if name != skip:
            text = step(text)
    return re.sub(r'\s+', ' ', text).strip()

def top_words(label, skip=None, n=15):
    sms = sms_original[data_spam['label'] == label]
    tokens = sms.apply(lambda t: remove_stopwords(run_pipeline(t, skip)))
    counts = Counter(w for toks in tokens for w in toks)
    return [w for w, _ in counts.most_common(n)]

# 1-2. The full pipeline (it gives the same result as preprocess())
for label in ['spam', 'ham']:
    print(label, top_words(label))

# 3. Skip one step at a time: which words enter the spam top 15?
base = top_words('spam')
print()
for name in steps:
    new = top_words('spam', skip=name)
    print(f"without {name:12}: {len(set(new) - set(base))} new words {sorted(set(new) - set(base))}")

**What to look for:**
- The spam words are about money and prizes (`free`, `claim`, `prize`, `call`, `txt`); the ham words are everyday chat (`u`, `go`, `say`). These are the words a classifier will use (Lab 2).
- `£` is in the spam top 15: `string.punctuation` does not contain it, so `remove_punctuation()` keeps it. Here it is useful!
- Without `lower`, the same word is counted more times: `Free`, `FREE`, `free`.
- Without `punctuation`, the symbols (`!`, `.`, `&`) enter the list.

# TIME for Exercise 1.2: recap of Lab 1_1

**How much does each step shrink the vocabulary?**

Start from `sms_original` and apply the steps one after the other (each one on top of the previous ones):

`lower` → `links` → `html` → `punctuation` → `numbers` → `repetitions` → `stop words`, then `stemming` and `lemmatization` on the result.

1. After each step count, with `WordPunctTokenizer`:
   - the **tokens**: how many words in total;
   - the **vocabulary**: how many *different* words.
2. Put the results in a table: `step | tokens | vocabulary`.
3. Find 3 groups of *different* words that the stemmer turns into the same stem.

Answer the questions:

>A. Which step reduces the vocabulary the most? \\
>B. Which step reduces the tokens a lot, but the vocabulary only a little? Why? \\
>C. Do stemming and lemmatization change the number of tokens? And the vocabulary? \\
>D. Are the groups you found in point 3 good merges? \\
>E. Is a smaller vocabulary always better? Give an example from the SMS where a step loses information useful to recognise spam.

*Hint: you can reuse the `steps` dictionary of Exercise 1.1.*

In [ ]:
# add your code here

### Solution 1.2

In [ ]:
from collections import Counter
from nltk.tokenize import WordPunctTokenizer

tokenizer = WordPunctTokenizer()

def stats(token_lists):
    tokens = [w for toks in token_lists for w in toks]
    return len(tokens), len(set(tokens))

rows = []
texts = sms_original.copy()
rows.append(('original', *stats(texts.apply(tokenizer.tokenize))))

# 1. One step at a time, each one applied on top of the previous ones
for name, step in steps.items():
    texts = texts.apply(step)
    rows.append((name, *stats(texts.apply(tokenizer.tokenize))))

tokens = texts.apply(remove_stopwords)
rows.append(('stop words', *stats(tokens)))
rows.append(('stemming', *stats(tokens.apply(stemming))))
rows.append(('lemmatization', *stats(tokens.apply(lemmatizer))))

table = pd.DataFrame(rows, columns=['step', 'tokens', 'vocabulary'])
table['vocabulary lost'] = (table['vocabulary'].shift() - table['vocabulary']).fillna(0).astype(int)
display(table)

# 2. Which different words does the stemmer merge into the same stem?
vocab = {w for toks in tokens for w in toks}
by_stem = {}
for w in vocab:
    by_stem.setdefault(porter_stemmer.stem(w), set()).add(w)
merged = sorted([(s, ws) for s, ws in by_stem.items() if len(ws) > 1], key=lambda kv: -len(kv[1]))
for stem, words in merged[:10]:
    print(f"{stem:10} <- {sorted(words)}")

**Answers:**
- **B.** `punctuation`: there are thousands of `.`, `,`, `!` tokens, but they are only a few *different* symbols. Removing them lowers the tokens a lot and the vocabulary a little.
- **C.** The tokens stay the same: stemming and lemmatization change each word, they do not remove any. The vocabulary shrinks, because different words become the same one (`mobile`, `mobiles` → `mobil`).
- **E.** For example `numbers`: `2day`, `4u`, `b4` become `day`, `u`, `b`, but they are typical of spam. Or `stop words`: `not` is removed, but it changes the meaning of the sentence.